In [ ]:
from pathlib import Path

import polars as pl

project_path = Path.cwd() / "output" / "project"

In [ ]:
1566 + 2960 + 4314565 - 4330336

-11245

In [ ]:
patient = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/patients.csv.gz")
patient.filter(pl.col("subject_id") == 10000032).collect()

subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
i64,str,i64,i64,str,str
10000032,"""F""",52,2180,"""2014 - 2016""","""2180-09-09"""


In [ ]:
icu_stay = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz")


# Creatinine

In [ ]:
6634093860 - 1648126260 

4985967600

In [ ]:
6637933500 - 1651965900

4985967600

In [ ]:
365.25 * 4

1461.0

In [ ]:
4985967600 / (1461 * 24 * 3600) * 4

157.99577914670317

In [ ]:
158

158

In [ ]:
labevent = pl.scan_parquet(project_path / "workspace" / "extraction" / "mimic-iv" / "3.1" / "labevents" / "labevent.parquet")
data = labevent.filter(pl.col("code").str.contains("Creatinine//")).collect()

FileNotFoundError: No such file or directory (os error 2): ...nICU-Example/output/project/workspace/extraction/mimic-iv/3.1/labevents/labevent.parquet (set POLARS_VERBOSE=1 to see full path)

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'filter'
	[3] 'sink'


In [ ]:
labevent.filter

<bound method LazyFrame.filter of <LazyFrame at 0x7CCE76BF1BA0>>

In [ ]:
data.filter(pl.col("subject_id")  == 10000032)

subject_id,time,code,numeric_value,text_value,hadm_id,stay_id
i64,datetime[μs],str,f32,str,str,str
10000032,2180-03-23 11:51:00,"""mimic-iv//3.1//labevents//labe…",0.4,"""0.4""",null,"""-1"""
10000032,2180-05-06 22:25:00,"""mimic-iv//3.1//labevents//labe…",0.3,"""0.3""",null,"""-1"""
10000032,2180-05-07 05:05:00,"""mimic-iv//3.1//labevents//labe…",0.3,"""0.3""","""22595853""","""-1"""
10000032,2180-06-03 12:00:00,"""mimic-iv//3.1//labevents//labe…",0.4,"""0.4""",null,"""-1"""
10000032,2180-06-03 12:00:00,"""mimic-iv//3.1//labevents//labe…",0.4,"""0.4""",null,"""-1"""
…,…,…,…,…,…,…
10000032,2180-08-05 21:20:00,"""mimic-iv//3.1//labevents//labe…",0.6,"""0.6""",null,"""-1"""
10000032,2180-08-06 06:36:00,"""mimic-iv//3.1//labevents//labe…",0.6,"""0.6""","""25742920""","""-1"""
10000032,2180-08-07 00:59:00,"""mimic-iv//3.1//labevents//labe…",0.4,"""0.4""","""25742920""","""-1"""


In [ ]:
for e in data.filter(pl.col("numeric_value").is_null()).iter_rows():
    print(e)

(17720947, datetime.datetime(2178, 10, 20, 12, 56), 'mimic-iv//3.1//labevents//labevent//Creatinine//mg/dL', None, '___', None, '-1')


In [ ]:
crea_directory = project_path / "workspace" /"concept" /"creatinine" / "1.0.0"

crea_paths = [str(p) for p in crea_directory.iterdir() if p.is_file()]

crea =  pl.scan_parquet(crea_paths[0])

In [ ]:
df = crea.collect()

In [ ]:
df

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,datetime[μs],str,f32,str,str,str,str
10000032,2180-03-23 11:51:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-06 22:25:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-07 05:05:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
…,…,…,…,…,…,…,…
19999987,2145-11-05 06:10:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-06 10:07:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-07 06:00:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
df = df.with_columns(
    (pl.col("time").dt.epoch("s")).alias("seconds_since_epoch")
)

In [ ]:
year = patient.filter(pl.col("subject_id") == 10000032).collect()["anchor_year"][0]

In [ ]:
df.sort("seconds_since_epoch").filter(pl.col("subject_id") == 10000032)

subject_id,time,code,numeric_value,text_value,dataset,table,src_code,seconds_since_epoch
i64,datetime[μs],str,f32,str,str,str,str,i64
10000032,2180-03-23 11:51:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6634093860
10000032,2180-05-06 22:25:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6637933500
10000032,2180-05-07 05:05:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6637957500
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6640315200
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6640315200
…,…,…,…,…,…,…,…,…
10000032,2180-08-05 21:20:00,"""creatinine//mg/dL""",0.6,"""0.6""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6645792000
10000032,2180-08-06 06:36:00,"""creatinine//mg/dL""",0.6,"""0.6""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6645825360
10000032,2180-08-07 00:59:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",6645891540


In [ ]:
(6634093860 - 1648126260 ) / (365.25 * 24 * 3600)

157.99577914670317

In [ ]:
1566+2960+4314565

4319091

In [ ]:
crea.collect()

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,datetime[μs],str,f32,str,str,str,str
10000032,2180-03-23 11:51:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-06 22:25:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-07 05:05:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
…,…,…,…,…,…,…,…
19999987,2145-11-05 06:10:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-06 10:07:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-07 06:00:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
crea.filter(pl.col("subject_id") == 12466550).collect()

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,datetime[μs],str,f32,str,str,str,str
12466550,2174-09-29 10:16:00,"""creatinine//mg/dL""",1.2,"""1.2""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-09-29 15:37:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-09-30 03:34:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-10-01 05:35:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-10-02 07:20:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
…,…,…,…,…,…,…,…
12466550,2174-10-07 06:20:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-10-08 09:15:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
12466550,2174-10-09 10:30:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
crea_subject = crea.filter(pl.col("subject_id") == 12466550).collect()
value = crea_subject[0]["time"][0]
charttime = value.second + value.minute * 60 + value.hour * 3600 + value.day * 3600 * 24 + value.month * 3600 * 24 * 30 + value.year * 3600 * 24 * 365
charttime

68585134560

In [ ]:
(55500 - 12480 ) / 60 /60

11.95

In [ ]:
60 * 95

5700

In [ ]:
str((12480/60 - (12480/60)%60)/60) + "h " + str((12480/60)%60) + "min"

'3.0h 28.0min'

In [ ]:
crea_subject.with_columns(((pl.col("time") - crea_subject[0]["time"][0])).alias("charttime"))

subject_id,time,code,numeric_value,text_value,dataset,table,src_code,charttime
i64,datetime[μs],str,f32,str,str,str,str,duration[μs]
12466550,2174-09-29 10:16:00,"""creatinine//mg/dL""",1.2,"""1.2""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",0µs
12466550,2174-09-29 15:37:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",5h 21m
12466550,2174-09-30 03:34:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",17h 18m
12466550,2174-10-01 05:35:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",1d 19h 19m
12466550,2174-10-02 07:20:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",2d 21h 4m
…,…,…,…,…,…,…,…,…
12466550,2174-10-07 06:20:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",7d 20h 4m
12466550,2174-10-08 09:15:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",8d 22h 59m
12466550,2174-10-09 10:30:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",10d 14m


In [ ]:
icustay = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz", schema_overrides={"intime":pl.Datetime}).filter(pl.col("subject_id") == 12466550).collect()
icustay

subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
i64,i64,i64,str,str,datetime[μs],str,f64
12466550,23998182,30000153,"""Trauma SICU (TSICU)""","""Trauma SICU (TSICU)""",2174-09-29 12:09:00,"""2174-10-01 03:26:10""",1.636921
12466550,23998182,31460471,"""Trauma SICU (TSICU)""","""Trauma SICU (TSICU)""",2174-10-04 19:15:28,"""2174-10-06 20:43:25""",2.061076


In [ ]:
icustay[0]["intime"][0]

datetime.datetime(2174, 9, 29, 12, 9)

In [ ]:
crea_subject.with_columns(((pl.col("time") - icustay[0]["intime"][0])).alias("charttime"))


subject_id,time,code,numeric_value,text_value,dataset,table,src_code,charttime
i64,datetime[μs],str,f32,str,str,str,str,duration[μs]
12466550,2174-09-29 10:16:00,"""creatinine//mg/dL""",1.2,"""1.2""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",-1h -53m
12466550,2174-09-29 15:37:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",3h 28m
12466550,2174-09-30 03:34:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",15h 25m
12466550,2174-10-01 05:35:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",1d 17h 26m
12466550,2174-10-02 07:20:00,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",2d 19h 11m
…,…,…,…,…,…,…,…,…
12466550,2174-10-07 06:20:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",7d 18h 11m
12466550,2174-10-08 09:15:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",8d 21h 6m
12466550,2174-10-09 10:30:00,"""creatinine//mg/dL""",1.1,"""1.1""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…",9d 22h 21m


In [ ]:
3 * 60 * 60 + 28 * 60

12480

In [ ]:
pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz").filter(pl.col("subject_id") == 12466550).collect()

subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
i64,i64,i64,str,str,str,str,f64
12466550,23998182,30000153,"""Trauma SICU (TSICU)""","""Trauma SICU (TSICU)""","""2174-09-29 12:09:00""","""2174-10-01 03:26:10""",1.636921
12466550,23998182,31460471,"""Trauma SICU (TSICU)""","""Trauma SICU (TSICU)""","""2174-10-04 19:15:28""","""2174-10-06 20:43:25""",2.061076


In [ ]:
for e in pl.scan_parquet("/workspaces/OpenICU-Example/output/project/datasets/extraction/metadata/codes.parquet").filter(pl.col("code").str.contains(".*?(Creatinine)//mg/dL") ).collect()["code"]:
    print(e)

mimic-iv//3.1//labevents//labevent//Creatinine//mg/dL
mimic-iv//3.1//labevents//labevent//Urine Creatinine//mg/dL


In [ ]:
x = crea.collect()

In [ ]:
x["code"].unique()

code
str
"""creatinine//mg/dL"""


In [ ]:
crea = crea.collect()

In [ ]:
crea

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,datetime[μs],str,f32,str,str,str,str
10000032,2180-03-23 11:51:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-06 22:25:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-07 05:05:00,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-06-03 12:00:00,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
…,…,…,…,…,…,…,…
19999987,2145-11-05 06:10:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-06 10:07:00,"""creatinine//mg/dL""",1.0,"""1.0""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
19999987,2145-11-07 06:00:00,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
crea = crea.with_columns(pl.col("time").dt.replace_time_zone("UTC"))

In [ ]:
crea = crea.filter(~pl.col("numeric_value").is_null(), pl.col("numeric_value").is_between(0, 15))

In [ ]:
ricu_crea = pl.read_parquet("./data/crea.parquet")

In [ ]:
ricu_crea

subject_id,charttime,crea,birthdate,charttime_origin
i32,duration[ms],f64,"datetime[μs, UTC]","datetime[μs, UTC]"
10000032,19075d 12h 51m,0.4,2128-01-01 00:00:00 UTC,2180-03-23 11:51:00 UTC
10000032,19119d 23h 25m,0.3,2128-01-01 00:00:00 UTC,2180-05-06 22:25:00 UTC
10000032,19120d 6h 5m,0.3,2128-01-01 00:00:00 UTC,2180-05-07 05:05:00 UTC
10000032,19147d 13h,0.4,2128-01-01 00:00:00 UTC,2180-06-03 12:00:00 UTC
10000032,19147d 13h,0.4,2128-01-01 00:00:00 UTC,2180-06-03 12:00:00 UTC
…,…,…,…,…
19999987,21127d 7h 10m,0.9,2088-01-01 00:00:00 UTC,2145-11-05 06:10:00 UTC
19999987,21128d 11h 7m,1.0,2088-01-01 00:00:00 UTC,2145-11-06 10:07:00 UTC
19999987,21129d 7h,0.9,2088-01-01 00:00:00 UTC,2145-11-07 06:00:00 UTC


In [ ]:
crea.head(2)

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,"datetime[μs, UTC]",str,f32,str,str,str,str
10000032,2180-03-23 11:51:00 UTC,"""creatinine//mg/dL""",0.4,"""0.4""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10000032,2180-05-06 22:25:00 UTC,"""creatinine//mg/dL""",0.3,"""0.3""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
x = pl.scan_parquet("output/project/datasets/extraction/data/labevents/labevent.parquet").filter(pl.col("subject_id") == 10026754).collect()
x["src_code"][0]

In [ ]:
x = crea.join(ricu_crea, right_on=["subject_id", "charttime_origin"], left_on=["subject_id", "time"], how="anti")
x

'mimic-iv//3.1//labevents//labevent//Creatinine//mg/dL'

In [ ]:
x.unique("text_value")

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,"datetime[μs, UTC]",str,f32,str,str,str,str
10026754,2140-03-15 11:10:00 UTC,"""creatinine//mg/dL""",0.8,"""___""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
crea.filter(pl.col("subject_id") == 10001725)

subject_id,time,code,numeric_value,text_value,dataset,table,src_code
i64,"datetime[μs, UTC]",str,f32,str,str,str,str
10001725,2110-04-11 18:02:00 UTC,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2110-04-12 02:59:00 UTC,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2110-04-13 06:00:00 UTC,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2110-04-14 07:10:00 UTC,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2110-05-04 17:17:00 UTC,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
…,…,…,…,…,…,…,…
10001725,2119-05-19 11:15:00 UTC,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2119-05-19 11:15:00 UTC,"""creatinine//mg/dL""",0.8,"""0.8""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"
10001725,2119-06-30 10:40:00 UTC,"""creatinine//mg/dL""",0.9,"""0.9""","""mimic-iv""","""labevents""","""mimic-iv//3.1//labevents//labe…"


In [ ]:
ricu_crea.join(crea, left_on=["subject_id", "charttime_origin"], right_on=["subject_id", "time"], how="anti")

subject_id,charttime,crea,birthdate,charttime_origin
i32,duration[ms],f64,"datetime[μs, UTC]","datetime[μs, UTC]"
12458810,27069d 2h 43m,1.0,2057-01-01 00:00:00 UTC,2131-02-12 01:43:00 UTC
